In [1]:
import os
import sys
import pandas as pd

# 将项目根目录加入 Python 路径（解决 ModuleNotFoundError）
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        print(f"✅ 已添加项目根目录: {project_root}")
else:
    # 如果当前已经在根目录，直接添加当前目录
    if os.path.exists(os.path.join(current_dir, 'config.py')):
        sys.path.insert(0, current_dir)
        print(f"✅ 当前目录已是项目根目录: {current_dir}")
    else:
        print("⚠️ 请确保当前工作目录在项目根目录或 notebooks 文件夹下")

from config import get_driver
from src.retriever import find_matching_phages, find_similar_cases

print("✅ 所有模块导入成功")

✅ 已添加项目根目录到路径: D:\IdeaProjects\vs\phage-workspace-mvp


In [3]:
# 查询 E. coli 的噬菌体（证据等级按 L1~L5 排序）
species = "Escherichia coli"
resistance = "MDR"

phages = find_matching_phages(driver, species, resistance, limit=20)

print(f"找到 {len(phages)} 个匹配噬菌体：\n")
for p in phages[:5]:  # 只显示前5条
    print(f"  {p['name']} (L{p['evidence_level']}) 概率: {p['infection_probability']} 来源: {p['evidence_ref']}")

找到 14 个匹配噬菌体：

  CP-p-BC-23086 (LL3) 概率: 0.98 来源: ['CASE-001']
  CP-p-BC-23062 (LL3) 概率: 0.96 来源: ['CASE-001']
  vB_EcoM_006 (LL2) 概率: 0.89 来源: []
  vB_EcoM_003 (LL2) 概率: 0.88 来源: []
  vB_EcoM_012 (LL2) 概率: 0.79 来源: []


In [4]:
# 查询 CRAB VAP 相似病例
species = "Acinetobacter baumannii"
infection_type = "VAP"

cases = find_similar_cases(driver, species, infection_type, limit=5)

print(f"找到 {len(cases)} 个相似病例：\n")
for c in cases:
    print(f"  {c['case_id']}: {c['infection_type']} 部位: {c['infection_site']} 结局: {c['clinical_outcome']}")
    print(f"    使用的噬菌体: {c['phages_used']}")
    print(f"    年龄组: {c.get('patient_age_group', 'N/A')}, 基础病: {c.get('comorbidities', [])}")
    print()

找到 2 个相似病例：

  CASE-009: VAP 部位: Lung 结局: Not improved
    使用的噬菌体: []
    年龄组: 55-65, 基础病: ['DM']

  CASE-002: VAP 部位: Lung 结局: Not improved
    使用的噬菌体: ['vB_AbaM_001']
    年龄组: 65-75, 基础病: ['COPD', 'DM']



In [5]:
# 验证数据库中的节点数
with driver.session() as session:
    result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count")
    print("数据库统计：")
    for record in result:
        print(f"  {record['label']}: {record['count']}")

数据库统计：
  Phage: 30
  Pathogen: 4
  PhageHostInteraction: 30
  ClinicalCase: 15
